In [1]:
import pandas as pd

stations = pd.read_csv("data/stations.csv")
days=pd.read_csv("data/station_day.csv",parse_dates=["Date"])
blr=stations[stations["City"]=="Bengaluru"]
df=days[days["StationId"].isin(blr["StationId"])]

df=df[["StationId", "Date", "AQI"]].dropna()
df=df.sort_values(["StationId","Date"])

print(df.shape)
df.head()

(9639, 3)


,StationId,Date,AQI
57267,KA002,2018-01-10,71.0
57268,KA002,2018-01-11,97.0
57269,KA002,2018-01-12,228.0
57270,KA002,2018-01-13,89.0
57271,KA002,2018-01-14,100.0


In [2]:
# 1) one row per calendar day for every station (missing days become NaN)
full = (df.set_index("Date")
          .groupby("StationId")["AQI"]
          .apply(lambda s: s.asfreq("D"))
          .reset_index())

g = full.groupby("StationId")["AQI"]

# 2) features: what we know by the end of "today"
full["lag1"] = g.shift(1)    # yesterday
full["lag2"] = g.shift(2)    # 2 days ago
full["lag3"] = g.shift(3)    # 3 days ago
full["mean7"] = g.transform(lambda s: s.shift(1).rolling(7).mean())  # avg of the 7 days before today
full["month"] = full["Date"].dt.month
full["weekday"] = full["Date"].dt.dayofweek

# 3) target: what we want to predict
full["target"] = g.shift(-1)  # tomorrow's AQI

data = full.dropna()
print(full.shape, data.shape)
data.head()

(11791, 10) (7162, 10)


,StationId,Date,AQI,lag1,lag2,lag3,mean7,month,weekday,target
7,KA002,2018-01-17,116.0,108.0,121.0,100.0,116.285714,1,2,109.0
8,KA002,2018-01-18,109.0,116.0,108.0,121.0,122.714286,1,3,117.0
9,KA002,2018-01-19,117.0,109.0,116.0,108.0,124.428571,1,4,129.0
10,KA002,2018-01-20,129.0,117.0,109.0,116.0,108.571429,1,5,125.0
11,KA002,2018-01-21,125.0,129.0,117.0,109.0,114.285714,1,6,113.0


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# station ID as a number, so the model can learn station differences
station_codes = {sid: i for i, sid in enumerate(sorted(data["StationId"].unique()))}
data = data.copy()
data["station"] = data["StationId"].map(station_codes)

features = ["AQI", "lag1", "lag2", "lag3", "mean7", "month", "weekday", "station"]

# time-based split: learn from the past, test on later dates (pre-lockdown)
train = data[data["Date"] < "2019-10-01"]
test = data[(data["Date"] >= "2019-10-01") & (data["Date"] < "2020-03-15")].copy()

model = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)
model.fit(train[features], train["target"])

test["pred"] = model.predict(test[features])

print("Train rows:", len(train), "| Test rows:", len(test))
print("Model MAE:   ", round(mean_absolute_error(test["target"], test["pred"]), 1))
print("Baseline MAE:", round(mean_absolute_error(test["target"], test["AQI"]), 1), "(tomorrow = today)")

per_station = test.groupby("StationId").apply(lambda t: pd.Series({
    "model_mae": mean_absolute_error(t["target"], t["pred"]),
    "baseline_mae": mean_absolute_error(t["target"], t["AQI"]),
    "test_days": len(t),
})).round(1)
per_station

Train rows: 5178 | Test rows: 1212
Model MAE:    15.4
Baseline MAE: 14.4 (tomorrow = today)


C:\Users\Shreya\AppData\Local\Temp\ipykernel_13088\4204296563.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_station = test.groupby("StationId").apply(lambda t: pd.Series({


,model_mae,baseline_mae,test_days
StationId,,,
KA002,18.6,17.2,121.0
KA003,27.0,27.0,117.0
KA004,19.4,20.2,107.0
KA005,7.1,5.3,40.0
KA006,14.3,13.3,147.0
KA007,14.5,13.2,126.0
KA008,12.3,11.6,148.0
KA009,12.6,11.9,157.0
KA010,11.5,8.0,124.0


In [6]:
from sklearn.linear_model import Ridge

# A) Random Forest predicting the CHANGE from today
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf.fit(train[features], train["target"] - train["AQI"])
test["pred_rf"] = test["AQI"] + rf.predict(test[features])

# B) Linear model on recent AQI only
lin_feats = ["AQI", "lag1", "lag2", "lag3", "mean7"]
lin = Ridge(alpha=1.0)
lin.fit(train[lin_feats], train["target"])
test["pred_lin"] = lin.predict(test[lin_feats])

for name, col in [("Baseline (tomorrow = today)", "AQI"),
                  ("Random Forest (raw value)", "pred"),
                  ("Random Forest (change)", "pred_rf"),
                  ("Linear (Ridge)", "pred_lin")]:
    print(f"{name:30s} MAE = {mean_absolute_error(test['target'], test[col]):.1f}")

Baseline (tomorrow = today)    MAE = 14.4
Random Forest (raw value)      MAE = 15.4
Random Forest (change)         MAE = 14.8
Linear (Ridge)                 MAE = 14.8


In [7]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# C) Gradient Boosting trained directly on absolute error, predicting the change
gb = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, random_state=42)
gb.fit(train[features], train["target"] - train["AQI"])
test["pred_gb"] = test["AQI"] + gb.predict(test[features])

rows = []
for name, col in [("Baseline (tomorrow = today)", "AQI"),
                  ("Random Forest (change)", "pred_rf"),
                  ("Linear (Ridge)", "pred_lin"),
                  ("Gradient Boosting (abs error)", "pred_gb")]:
    rows.append({"model": name,
                 "MAE": mean_absolute_error(test["target"], test[col]),
                 "RMSE": mean_squared_error(test["target"], test[col]) ** 0.5})
pd.DataFrame(rows).round(1)

,model,MAE,RMSE
0,Baseline (tomorrow = today),14.4,25.4
1,Random Forest (change),14.8,24.5
2,Linear (Ridge),14.8,24.7
3,Gradient Boosting (abs error),13.8,23.2


In [10]:
import json, os

# Demo snapshot: 14 Mar 2020 = last normal day before the COVID lockdown.
# We use `gb` (trained only on data before Oct 2019), so the prediction for 15 Mar
# is a genuine out-of-sample forecast we can compare with what actually happened.
DEMO_DATE = "2020-03-14"

# approximate station coordinates (lat, lon)
coords = {
    "KA002": (12.9135, 77.5951), "KA003": (12.9351, 77.6813), "KA004": (12.9518, 77.5398),
    "KA005": (12.9779, 77.5698), "KA006": (13.0294, 77.5856), "KA007": (12.9385, 77.5958),
    "KA008": (12.9205, 77.5847), "KA009": (13.0274, 77.4945), "KA010": (12.9895, 77.5427),
    "KA011": (12.9175, 77.6229),
}
names = dict(zip(blr["StationId"], blr["StationName"].str.replace(r",\s*Bengaluru.*", "", regex=True)))

def category(aqi):
    for limit, label in [(50, "Good"), (100, "Satisfactory"), (200, "Moderate"),
                         (300, "Poor"), (400, "Very Poor")]:
        if aqi <= limit:
            return label
    return "Severe"

# each station's latest reading on or before the demo date -> predict the next day
rows = full[full["Date"] <= DEMO_DATE].dropna(subset=["AQI"]).copy()
rows["station"] = rows["StationId"].map(station_codes)
latest = rows.sort_values("Date").groupby("StationId").tail(1)
latest_pred = latest["AQI"].values + gb.predict(latest[features])

out = []
for (_, r), p in zip(latest.iterrows(), latest_pred):
    sid = r["StationId"]
    hist = df[(df["StationId"] == sid) & (df["Date"] <= r["Date"])].tail(7)
    t = test[test["StationId"] == sid]
    actual = r["target"]   # what really happened the next day (NaN if not recorded)
    out.append({
        "id": sid, "name": names[sid], "lat": coords[sid][0], "lon": coords[sid][1],
        "latest_date": r["Date"].strftime("%Y-%m-%d"), "latest_aqi": round(float(r["AQI"])),
        "latest_category": category(r["AQI"]),
        "pred_date": (r["Date"] + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        "pred_aqi": round(float(p)), "pred_category": category(p),
        "actual_next": None if pd.isna(actual) else round(float(actual)),
        "actual_next_category": None if pd.isna(actual) else category(actual),
        "history": [{"date": d.strftime("%Y-%m-%d"), "aqi": round(float(a))}
                    for d, a in zip(hist["Date"], hist["AQI"])],
        "model_mae": round(mean_absolute_error(t["target"], t["pred_gb"]), 1),
        "baseline_mae": round(mean_absolute_error(t["target"], t["AQI"]), 1),
        "test_days": len(t),
    })

summary = {
    "model": "Gradient Boosting (absolute error)",
    "demo_date": DEMO_DATE,
    "test_period": "1 Oct 2019 - 14 Mar 2020 (pre-lockdown)",
    "model_mae": round(mean_absolute_error(test["target"], test["pred_gb"]), 1),
    "baseline_mae": round(mean_absolute_error(test["target"], test["AQI"]), 1),
    "model_rmse": round(mean_squared_error(test["target"], test["pred_gb"]) ** 0.5, 1),
    "baseline_rmse": round(mean_squared_error(test["target"], test["AQI"]) ** 0.5, 1),
}

payload = {"summary": summary, "stations": out}
os.makedirs("site", exist_ok=True)
with open("site/data.json", "w") as f:
    json.dump(payload, f, indent=2)
with open("site/data.js", "w") as f:   # lets the page work even when opened as a local file
    f.write("window.NAMMA_GAALI_DATA = " + json.dumps(payload) + ";\n")

print("Saved site/data.json + site/data.js with", len(out), "stations")
pd.DataFrame(out)[["id", "name", "latest_date", "latest_aqi", "pred_aqi", "actual_next", "pred_category", "model_mae", "baseline_mae"]]

Saved site/data.json + site/data.js with 10 stations


,id,name,latest_date,latest_aqi,pred_aqi,actual_next,pred_category,model_mae,baseline_mae
0,KA005,City Railway Station,2020-03-13,97,105,NaN,Moderate,6.2,5.3
1,KA008,Jayanagar 5th Block,2020-03-14,86,86,87.0,Satisfactory,11.3,11.6
2,KA007,Hombegowda Nagar,2020-03-14,96,97,104.0,Satisfactory,13.1,13.2
3,KA002,BTM Layout,2020-03-14,81,79,80.0,Satisfactory,15.7,17.2
4,KA009,Peenya,2020-03-14,83,81,73.0,Satisfactory,10.2,11.9
5,KA010,Sanegurava Halli,2020-03-14,69,72,67.0,Satisfactory,8.3,8.0
6,KA004,Bapuji Nagar,2020-03-14,99,99,110.0,Satisfactory,19.9,20.2
7,KA003,BWSSB Kadabesanahalli,2020-03-14,95,99,81.0,Satisfactory,25.7,27.0
8,KA006,Hebbal,2020-03-14,87,90,100.0,Satisfactory,12.7,13.3
9,KA011,Silk Board,2020-03-14,83,86,76.0,Satisfactory,13.0,13.5
